# Persistent trajectory learning for predictive AHR

This experiment notebook supplies exact-rule tactical curricula, immutable complete game trajectories, eight board symmetries, stratified replay, sequence prediction examples and an interruption-safe AHR trainer. The control and predictive arms use identical model initialization, curricula, supervised objectives and declared update/game budgets. The predictive arm adds the action-conditioned latent objective.

**Inputs:** original notebook definitions, the selected `go09-first-selfplay-v1` checkpoint, versioned model notebook 08 and a frozen curriculum. **Outputs:** curriculum data, full trajectories and manifests, checkpoints under `checkpoints/<run_id>/`, and JSON receipts under `reports/ahr_jepa_v3/`. All imports execute only tagged `ahr-export` definitions.

**Runtime:** default Run All performs bounded data/checkpoint checks; full training is explicit and may take days. The controller launches training in detached tmux. **Resume:** rerun the same controller stage or call `v3_run_training(PROJECT, run_id, V3TrainConfig(**the_original_config))`; do not change configuration or source after starting. The latest two valid full checkpoints and the best observed training-loss checkpoint are retained. Best training loss is not a playing-strength claim.

All terminal labels require two legal passes and independent score verification. UNKNOWN game prefixes contribute transitions only. Tactical statements are limited to their named one-, two- or three-ply predicates, never full-game optimality.

In [ ]:
from pathlib import Path
import os, json, nbformat
PROJECT=Path.cwd().resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').exists():
    if PROJECT.parent==PROJECT: raise RuntimeError('Run inside AHR Go workspace')
    PROJECT=PROJECT.parent
shared=nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4)
for cell in shared.cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):
        exec(compile(cell.source,'00_shared:ahr-export','exec'),globals())
v3_load_upstream(PROJECT,globals())
v3_import_exports(PROJECT/'experiments/ahr_jepa_v3/08_predictive_ahr.ipynb',globals())
torch.set_num_threads(1)


## Exact symmetry and tactical predicates
The same coordinate mapping transforms board, complete superko history, actions, policies and ownership. Pattern families are assigned to splits before augmentation; a family is the canonical local stone topology around the target, not a full-board hash.

In [ ]:
from dataclasses import dataclass, asdict, field
from collections import Counter, defaultdict
import copy, hashlib, json, math, os, random, time, uuid, subprocess, sys
import numpy as np
import torch

V3_TRAIN_SCHEMA='ahr_trajectory_trainer_v3'
V3_TRAJECTORY_SCHEMA='ahr_full_trajectory_v3'
V3_HORIZONS=(1,2,4,8)

def v3_action_transform(action,symmetry,size=9):
    if type(action) is not int or not 0<=action<=size*size or not 0<=symmetry<8:raise ValueError('Invalid action/symmetry')
    if action==size*size:return action
    row,col=divmod(action,size)
    if symmetry>=4:col=size-1-col
    for _ in range(symmetry%4):row,col=col,size-1-row
    return row*size+col

def v3_vector_transform(values,symmetry,size=9,with_pass=False):
    if len(values)!=size*size+int(with_pass):raise ValueError('Vector shape mismatch')
    result=[0.0]*len(values)
    for action,value in enumerate(values):result[v3_action_transform(action,symmetry,size)]=value
    return result

def v3_augment_example(example,symmetry):
    result=copy.deepcopy(example);state=from_dict(example['state']);size=state.size
    result['state']=to_dict(transform_go(state,symmetry))
    result['policy_target']=v3_vector_transform(example['policy_target'],symmetry,size,True)
    if 'ownership_target' in result:result['ownership_target']=v3_vector_transform(result['ownership_target'],symmetry,size)
    for key in ('targets','good_actions'):
        if key in result:result[key]=[v3_action_transform(int(a),symmetry,size) for a in result[key]]
    result['symmetry']=symmetry
    return result

def v3_augment_sequence(sequence,symmetry):
    result=copy.deepcopy(sequence);state=from_dict(sequence['state']);size=state.size
    result['state']=to_dict(transform_go(state,symmetry))
    result['future_state']=to_dict(transform_go(from_dict(sequence['future_state']),symmetry))
    result['actions']=[v3_action_transform(int(a),symmetry,size) for a in sequence['actions']]
    result['symmetry']=symmetry
    return result

def v3_pattern_family(state,targets,category):
    # Translation, color, rotation and reflection invariant local topology. Outside
    # board is an explicit third symbol, so corner/edge/interior are distinct.
    targets=set(targets);forms=[]
    anchors=sorted(targets) or [state.pass_action//2]
    for symmetry in range(8):
        transformed=transform_go(state,symmetry)
        transformed_targets={v3_action_transform(a,symmetry,state.size) for a in targets}
        for anchor in anchors:
            center=v3_action_transform(anchor,symmetry,state.size);row,col=divmod(center,state.size)
            patch=[]
            for dr in range(-2,3):
                for dc in range(-2,3):
                    rr,cc=row+dr,col+dc
                    if 0<=rr<state.size and 0<=cc<state.size:
                        point=rr*state.size+cc
                        patch.append((transformed.board[point]*state.to_play,int(point in transformed_targets)))
                    else:patch.append((2,0))
            forms.append(json.dumps(patch,separators=(',',':')))
    return v3_digest({'topology':min(forms),'radius':2,'category_independent':True})

def v3_family_split(family_id):
    bucket=int(hashlib.sha256(('family-split-v3:'+family_id).encode()).hexdigest()[:8],16)%10
    return 'development' if bucket==0 else 'lockbox' if bucket==1 else 'train'

def v3_tactical_success(case,action):
    if case['goal']!='connect_one_ply':return arena_tactical_success(case,action)
    state=from_dict(case['state']);child=arena_reference_child(state,action)
    if child is None:return False
    targets=case['targets'];color=state.to_play
    if any(child.board[p]!=color for p in targets):return False
    stones,_=arena_reference_group(child.board,child.size,targets[0])
    return set(targets)<=stones

def v3_tactical_candidates(state):
    groups=groups_and_liberties(state.board,state.size)
    friendly=[]
    for stones,liberties in groups:
        targets=sorted(stones);color=state.board[targets[0]]
        if color==state.to_play:
            friendly.append((set(stones),set(liberties)))
            if len(liberties)==1:
                yield {'category':'atari_escape','goal':'survive_two_plies','targets':targets,'candidates':sorted(legal_actions(state))}
        elif len(liberties)==1:
            yield {'category':'life_death_bounded' if len(stones)>=3 else 'capture','goal':'capture_one_ply','targets':targets,'candidates':sorted(liberties)}
        elif len(liberties)==2:
            yield {'category':'ladder_bounded','goal':'force_capture_three_plies','targets':targets,'candidates':sorted(liberties)}
    seen=set()
    for i,(left,left_libs) in enumerate(friendly):
        for right,right_libs in friendly[i+1:]:
            shared=left_libs&right_libs
            if shared:
                targets=(min(left),min(right))
                if targets not in seen:
                    seen.add(targets)
                    yield {'category':'connection','goal':'connect_one_ply','targets':list(targets),'candidates':sorted(shared)}

def v3_verify_sequence(sequence):
    if len(sequence['actions']) not in V3_HORIZONS:raise ValueError('Unexpected prediction horizon')
    state=from_dict(sequence['state'])
    for action in sequence['actions']:
        child=play(state,int(action))
        if not verify_go_emission(state,int(action),child):raise ValueError('Independent transition mismatch')
        state=child
    if to_dict(state)!=sequence['future_state']:raise ValueError('Future state/history mismatch')
    return True

def v3_read_curriculum(project,descriptor=None):
    path=Path(project)/'experiments/ahr_jepa_v3/data/curriculum.json' if descriptor is None else Path(project)/(descriptor['path'] if isinstance(descriptor,dict) else str(descriptor))
    desc_path=path.with_suffix('.manifest.json')
    desc=json.loads(desc_path.read_text())
    if file_sha256(path)!=desc['sha256']:raise RuntimeError('Curriculum hash mismatch')
    if isinstance(descriptor,dict) and descriptor['sha256']!=desc['sha256']:raise RuntimeError('Requested curriculum changed')
    data=json.loads(path.read_text())
    families=defaultdict(set)
    for example in data['examples']:families[example['family_id']].add(example['split'])
    if any(len(splits)!=1 for splits in families.values()):raise RuntimeError('Family leakage across splits')
    if any(e['split']!=v3_family_split(e['family_id']) for e in data['examples']):raise RuntimeError('Family split changed')
    return data,desc

def v3_generate_curriculum(project,seed=20260927,positions=384,max_games=160):
    project=Path(project);path=project/'experiments/ahr_jepa_v3/data/curriculum.json'
    config={'seed':seed,'positions':positions,'max_games':max_games,'schema':'v3-tactical-curriculum','split_unit':'canonical local target topology before augmentation','max_examples_per_game':12,'max_category_examples_per_game':4,'minimum_source_games':30,'category_independent_families':True}
    with checkpoint_lock(path.parent/'curriculum-writer'):
        if path.exists():
            data,descriptor=v3_read_curriculum(project)
            if data['generation_config']!=config:raise RuntimeError('Frozen curriculum configuration changed')
            return descriptor
        generation_dir=project/'checkpoints'/'ahr-v3-curriculum-generation'
        generation_identity={'config':config,'source':v3_code_sha(project/'experiments/ahr_jepa_v3/09_trajectory_learning.ipynb',True)}
        saved=None;manifest_file=generation_dir/'manifest.json'
        if manifest_file.exists():
            manifest=json.loads(manifest_file.read_text())
            for entry in manifest['entries']:
                payload=_checkpoint_entry_state(generation_dir,entry)
                if payload is None or payload.get('identity')!=generation_identity:raise RuntimeError('Curriculum generation checkpoint/source corrupt or changed')
                if entry['file']==manifest['latest']:saved=payload
        elif list(generation_dir.glob('*.pt')):raise RuntimeError('Curriculum checkpoint manifest missing')
        if saved is None:
            saved={'schema':'ahr-v3-curriculum-generation','identity':generation_identity,'config':config,'global_step':0,'next_game':0,
                   'model':None,'optimizer':None,'scheduler':None,'scaler':None,'rng':None,'sampler':{'seed':seed},'metrics':{},'examples':[],'sequences':[],'counts':{},'unknown':{}}
            _save_checkpoint_unlocked(generation_dir,saved,best=True,resume_cell='09: v3_generate_curriculum with identical arguments')
        examples=saved['examples'];sequences=saved['sequences'];families={e['family_id'] for e in examples};seen_positions=set();counts=Counter(saved['counts']);unknown=Counter(saved['unknown'])
        categories=('capture','atari_escape','connection','ladder_bounded','life_death_bounded')
        quota=max(20,math.ceil(positions/len(categories)))
        split_quota={'development':max(4,quota//6),'lockbox':max(4,quota//7)}
        split_quota['train']=quota-sum(split_quota.values())
        split_counts=Counter((e['category'],e['split']) for e in examples)
        legacy=arena_tactical_suite()['cases']
        excluded={v3_pattern_family(from_dict(c['state']),c['targets'],category) for c in legacy if c['targets'] for category in categories}
        # Existing diagnostics stay exclusively evaluation-only. Exclude their
        # local topology even if a matching pattern arises in a random game.
        for game_id in range(saved['next_game'],max_games):
            rng=random.Random(seed+1000003*game_id);local_counts=Counter()
            board=initial_state(9,7.5);states=[to_dict(board)];actions=[];new_examples=[]
            game_split=v3_family_split(v3_digest({'curriculum_game':game_id,'seed':seed}))
            for ply in range(110):
                legal=[a for a in legal_actions(board) if a!=81]
                if not legal:break
                action=rng.choice(legal);child=play(board,action)
                if not verify_go_emission(board,action,child):raise RuntimeError('Independent generation check failed')
                actions.append(action);board=child;states.append(to_dict(board))
                if ply<18 or ply%3 or len(new_examples)>=12:continue
                for case in v3_tactical_candidates(board):
                    category=case['category']
                    if split_counts[(category,game_split)]>=split_quota[game_split] or local_counts[category]>=4 or len(new_examples)>=12:continue
                    family=v3_pattern_family(board,case['targets'],category)
                    key=(canonical_go(board),case['goal'],tuple(case['targets']))
                    if family in excluded or family in families or key in seen_positions or v3_family_split(family)!=game_split:continue
                    case.update(state=to_dict(board))
                    good=[a for a in case.pop('candidates') if v3_tactical_success(case,a)]
                    if not good:
                        unknown[category]+=1
                        continue
                    policy=[0.0]*82
                    for a in good:policy[a]=1/len(good)
                    example={**case,'family_id':family,'split':v3_family_split(family),'policy_target':policy,'policy_weight':1.0,'good_actions':good,
                        'bounded_objective':{'kind':case['goal'],'status':'VERIFIED_POSITIVE_WITNESSES','horizon':3 if 'three' in case['goal'] else 2 if 'two' in case['goal'] else 1,'full_game_result':'UNKNOWN'},
                        'provenance':{'generator':'legal_random_trajectory_exact_predicate','seed':seed,'game_id':game_id,'ply':len(actions),'external_labels':False,'old_diagnostic_family':False}}
                    examples.append(example);new_examples.append((len(actions),example));families.add(family);seen_positions.add(key);counts[category]+=1;split_counts[(category,game_split)]+=1;local_counts[category]+=1
            # Prediction sequences inherit their source tactical family's split.
            # No sequence from a held-out family is admitted to training.
            for index,example in new_examples:
                for horizon in V3_HORIZONS:
                    if index+horizon<len(states):
                        sequence={'state':states[index],'future_state':states[index+horizon],'actions':actions[index:index+horizon],
                            'family_id':example['family_id'],'split':example['split'],'source':'curriculum_legal_transition','game_id':game_id,'start_ply':index}
                        v3_verify_sequence(sequence);sequences.append(sequence)
            saved.update(global_step=game_id+1,next_game=game_id+1,examples=examples,sequences=sequences,counts=dict(counts),unknown=dict(unknown),rng={'per_game_seed':seed+1000003*(game_id+1)})
            _save_checkpoint_unlocked(generation_dir,saved,best=True,resume_cell='09: v3_generate_curriculum with identical arguments')
            atomic_json(v3_report_path(project,'curriculum_progress.json'),{'status':'RUNNING','games':game_id+1,'examples':len(examples),'categories':dict(counts),'sequences':len(sequences)})
            if len({e['provenance']['game_id'] for e in examples})>=30 and all(split_counts[(c,split)]>=quota_for_split for c in categories for split,quota_for_split in split_quota.items()):break
        if len({e['provenance']['game_id'] for e in examples})<30 or any(split_counts[(c,split)]<4 for c in categories for split in split_quota):raise RuntimeError('Insufficient category/family split coverage: '+str(split_counts))
        # A game may generate multiple local families. To prevent future board
        # overlap, move every sequence from such a game to the strictest split.
        # Supervised examples retain whole-pattern splits; all sequences from a
        # game with development/lockbox examples are excluded from training.
        game_splits=defaultdict(set)
        for e in examples:game_splits[e['provenance']['game_id']].add(e['split'])
        for sequence in sequences:
            splits=game_splits[sequence['game_id']]
            sequence['split']='lockbox' if 'lockbox' in splits else 'development' if 'development' in splits else 'train'
        data={'schema':'v3-tactical-curriculum','generation_config':config,'examples':examples,'sequences':sequences,
              'categories':dict(counts),'rejected_no_positive_bounded_witness':dict(unknown),'excluded_old_pattern_families':sorted(excluded),
              'claims':'Bounded verified predicates only; no terminal outcome targets or full-game optimality labels',
              'split_policy':'Canonical local pattern families before D4; whole source games separated for prediction sequences; lockbox never used for training/development selection'}
        atomic_json(path,data)
        descriptor={'path':str(path.relative_to(project)),'sha256':file_sha256(path),'examples':len(examples),'sequences':len(sequences),
                    'splits':dict(Counter(e['split'] for e in examples)),'sequence_splits':dict(Counter(e['split'] for e in sequences)),
                    'categories':dict(counts),'generation_config':config,'family_count':len(families)}
        atomic_json(path.with_suffix('.manifest.json'),descriptor)
        return descriptor


## Immutable trajectories and stratified sampling
Each complete game or explicitly UNKNOWN prefix is replayed from its initial state and independently checked before admission. Immutable JSON payloads are addressed by SHA-256; a per-run manifest records their order. UNKNOWN trajectories cannot produce policy or result supervision.

In [ ]:
def v3_verify_trajectory(game):
    if game.get('schema')!=V3_TRAJECTORY_SCHEMA or game.get('split')!='train':raise ValueError('Invalid trajectory schema/split')
    state=from_dict(game['initial_state'])
    if not game['records']:raise ValueError('Empty trajectory')
    for record in game['records']:
        if record['state']!=to_dict(state):raise ValueError('Chronological state/history mismatch')
        action=record['action'];selfplay09_policy(state,record['policy_target']);behavior=selfplay09_policy(state,record['behavior_policy'])
        if behavior[action]<=0:raise ValueError('Actual move not in behavior distribution')
        child=play(state,action)
        if not verify_go_emission(state,action,child):raise ValueError('Independent move verifier rejected trajectory')
        state=child
    if to_dict(state)!=game['state']:raise ValueError('Final state mismatch')
    if game['status']=='VERIFIED_TERMINAL':
        if not state.terminal:raise ValueError('Nonterminal outcome label')
        if area_score(state)!=arena_reference_score(state):raise ValueError('Independent terminal scoring disagreement')
    elif game['status']=='UNKNOWN':
        if state.terminal:raise ValueError('Terminal trajectory mislabeled UNKNOWN')
    else:raise ValueError('Cannot archive unfinished status')
    return state

def v3_archive_root(project,run_id):
    if not isinstance(run_id,str) or Path(run_id).name!=run_id or run_id in ('','.','..'):raise ValueError('Simple run ID required')
    return Path(project)/'experiments/ahr_jepa_v3/data/trajectories'/run_id

def v3_archive_append(project,run_id,game):
    final=v3_verify_trajectory(game);root=v3_archive_root(project,run_id);root.mkdir(parents=True,exist_ok=True)
    payload=(json.dumps(game,sort_keys=True,separators=(',',':'),allow_nan=False)+'\n').encode();digest=hashlib.sha256(payload).hexdigest()
    manifest_path=root/'manifest.json';manifest=json.loads(manifest_path.read_text()) if manifest_path.exists() else {'schema':V3_TRAJECTORY_SCHEMA,'run_id':run_id,'entries':[]}
    for entry in manifest['entries']:
        if not (root/entry['file']).is_file() or file_sha256(root/entry['file'])!=entry['sha256']:raise RuntimeError('Existing immutable trajectory corrupt')
    existing=[e for e in manifest['entries'] if e['game_id']==game['game_id']]
    if existing:
        if existing[0]['sha256']!=digest:raise RuntimeError('Same game identity produced different trajectory')
        return existing[0]
    target=root/(digest+'.json')
    if target.exists() and file_sha256(target)!=digest:raise RuntimeError('Trajectory digest collision/corruption')
    if not target.exists():atomic_write_bytes(target,payload)
    entry={'game_id':game['game_id'],'file':target.name,'sha256':digest,'plies':len(game['records']),'status':game['status'],
        'outcome':('black' if area_score(final)>0 else 'white' if area_score(final)<0 else 'draw') if final.terminal else 'UNKNOWN','split':'train'}
    manifest['entries'].append(entry);atomic_json(manifest_path,manifest)
    return entry

def v3_archive_read(project,run_id,entry):
    path=v3_archive_root(project,run_id)/entry['file']
    if file_sha256(path)!=entry['sha256']:raise RuntimeError('Immutable trajectory hash mismatch')
    return json.loads(path.read_text())

def v3_verify_archive_references(project,run_id,entries):
    root=v3_archive_root(project,run_id);path=root/'manifest.json'
    if not entries:
        if path.exists() and json.loads(path.read_text())['entries']:
            # A crash can commit an immutable archive entry before the next model
            # checkpoint. It is verified/replayed by deterministic ingest later.
            return True
        return True
    manifest=json.loads(path.read_text());index={e['sha256']:e for e in manifest['entries']}
    for entry in entries:
        if index.get(entry['sha256'])!=entry:raise RuntimeError('Committed archive reference missing/changed')
        game=v3_archive_read(project,run_id,entry)
        if game['game_id']!=entry['game_id']:raise RuntimeError('Archive game identity mismatch')
    return True

def v3_trajectory_example(game,index):
    if game['status']!='VERIFIED_TERMINAL':raise ValueError('UNKNOWN prefixes are transition-only')
    record=game['records'][index];final=from_dict(game['state']);state=from_dict(record['state'])
    return {'state':copy.deepcopy(record['state']),'policy_target':list(record['policy_target']),
            'policy_weight':0.0 if record.get('fallback_pass',False) else 1.0,
            **terminal_go_targets(final,state.to_play),'game_id':game['game_id'],'source':'verified_ahr_trajectory','split':'train'}

def v3_trajectory_sequence(game,index,horizon):
    if horizon not in V3_HORIZONS or index+horizon>len(game['records']):raise ValueError('Sequence out of range')
    future=game['records'][index+horizon]['state'] if index+horizon<len(game['records']) else game['state']
    return {'state':copy.deepcopy(game['records'][index]['state']),'future_state':copy.deepcopy(future),
            'actions':[r['action'] for r in game['records'][index:index+horizon]],'game_id':game['game_id'],'split':'train','source':game['status']}

def v3_stratified_batch(project,run_id,state,config,curriculum,runtime):
    examples=[];sequences=[];sampler=state['sampler'];train_examples=[e for e in curriculum['examples'] if e['split']=='train']
    train_sequences=[e for e in curriculum['sequences'] if e['split']=='train']
    categories=sorted({e['category'] for e in train_examples});bycat={c:[e for e in train_examples if e['category']==c] for c in categories}
    entries=state['archive_entries'][-config.replay_capacity_games:];terminal=[e for e in entries if e['status']=='VERIFIED_TERMINAL']
    groups=defaultdict(list)
    for age,entry in enumerate(terminal):
        bucket='recent' if age>=max(0,len(terminal)*3//4) else 'older'
        groups[(bucket,entry['outcome'])].append(entry)
    strata=sorted(groups)
    cache=runtime.setdefault('games',{})
    def read(entry):
        if entry['sha256'] not in cache:
            if len(cache)>=32:cache.pop(next(iter(cache)))
            cache[entry['sha256']]=v3_archive_read(project,run_id,entry)
        return cache[entry['sha256']]
    for i in range(config.batch_size):
        cursor=sampler['position'];sampler['position']+=1
        tactical=not terminal or (i/config.batch_size)<config.tactical_fraction
        if tactical:
            category=categories[cursor%len(categories)];example=copy.deepcopy(random.choice(bycat[category]))
            sampler['category_counts'][category]=sampler['category_counts'].get(category,0)+1
        else:
            stratum=strata[cursor%len(strata)];entry=random.choice(groups[stratum]);game=read(entry)
            example=v3_trajectory_example(game,random.randrange(len(game['records'])))
            key='/'.join(stratum);sampler['archive_strata'][key]=sampler['archive_strata'].get(key,0)+1
        examples.append(v3_augment_example(example,random.randrange(8)))
        horizon=V3_HORIZONS[cursor%len(V3_HORIZONS)]
        eligible=[e for e in entries if e['plies']>=horizon]
        if eligible:
            entry=random.choice(eligible);game=read(entry);index=random.randrange(len(game['records'])-horizon+1)
            sequence=v3_trajectory_sequence(game,index,horizon)
        else:
            choices=[s for s in train_sequences if len(s['actions'])==horizon]
            if not choices:continue
            sequence=copy.deepcopy(random.choice(choices))
        sequences.append(v3_augment_sequence(sequence,random.randrange(8)))
        sampler['horizons'][str(horizon)]=sampler['horizons'].get(str(horizon),0)+1
    sampler['epoch']=sampler['position']//max(1,len(train_examples)+sum(e['plies'] for e in terminal))
    return examples,sequences


## Source-bound training and exact continuation
Events separate warmup updates, game creation, legal moves, immutable ingestion, optional deeper AHR target reanalysis, and optimizer updates. Checkpoints include both online/EMA networks, all optimizer state and RNG streams, sampler cursors, complete active-game history, immutable replay references and historical opponent weights. Actor caches can be rebuilt without consuming restored random streams.

In [ ]:
@dataclass(frozen=True)
class V3TrainConfig:
    seed:int=2026092701
    use_jepa:bool=False
    games:int=128
    warmup_updates:int=256
    updates_per_game:int=16
    batch_size:int=32
    learning_rate:float=0.0003
    weight_decay:float=0.0001
    device:str='cuda:0'
    actor_device:str='cuda:1'
    board_size:int=9
    komi:float=7.5
    max_plies:int=512
    time_budget_s:float=1.0
    search:dict=field(default_factory=lambda: {'max_depth':4,'max_nodes':192,'branch_width':8,'queen_cycles':2})
    queen_cycles:int=2
    temperature:float=1.0
    exploration:float=0.1
    tactical_fraction:float=0.25
    replay_capacity_games:int=4096
    snapshot_every_games:int=8
    opponent_probability:float=0.25
    opponent_snapshots:int=8
    ema_decay:float=0.99
    jepa_weight:float=0.1
    initial_run:str='go09-first-selfplay-v1'
    initial_selection:str='latest'
    initial_checkpoint_sha256:str='3e4287bc1e8885fa975660b07bc76800e22aa6f8627fd896cc3163280b1f95ad'
    curriculum_path:str='experiments/ahr_jepa_v3/data/curriculum.json'
    reanalysis_per_game:int=4
    reanalysis_search:dict=field(default_factory=lambda: {'max_depth':5,'max_nodes':384,'branch_width':12,'queen_cycles':2})
    reanalysis_time_budget_s:float=2.0
    checkpoint_every_events:int=32
    checkpoint_every_optimizer_steps:int=250
    checkpoint_interval_s:float=600.0
    purpose:str='go_training'
    validation_force_pass:bool=False
    def __post_init__(self):
        for name in ('games','updates_per_game','batch_size','max_plies','replay_capacity_games','snapshot_every_games','opponent_snapshots','checkpoint_every_events','checkpoint_every_optimizer_steps'):
            if type(getattr(self,name)) is not int or getattr(self,name)<1:raise ValueError(name+' must be positive integer')
        if self.warmup_updates<0 or self.reanalysis_per_game<0:raise ValueError('Nonnegative optional update counts required')
        if self.board_size!=9 or self.queen_cycles!=2:raise ValueError('This controlled experiment fixes 9x9 and two Queen cycles')
        if self.search.get('queen_cycles',2)!=self.queen_cycles or self.reanalysis_search.get('queen_cycles',2)!=self.queen_cycles:raise ValueError('Training/acting reasoning budget mismatch')
        if not 0<=self.tactical_fraction<=1 or not 0<=self.exploration<=1 or not 0<=self.opponent_probability<=1:raise ValueError('Mixture probability out of range')
        if not 0<self.ema_decay<1 or self.jepa_weight<0 or self.learning_rate<=0 or self.temperature<=0:raise ValueError('Invalid training scalar')
        if self.checkpoint_interval_s>600 or self.checkpoint_every_optimizer_steps>250 or self.checkpoint_every_events>32:raise ValueError('Checkpoint cadence too sparse')
        if self.purpose not in ('go_training','synthetic_smoke'):raise ValueError('Unknown training purpose')
        if self.validation_force_pass and self.purpose!='synthetic_smoke':raise ValueError('Synthetic fixtures cannot become trained candidates')
        V3SearchConfig(**self.search);V3SearchConfig(**self.reanalysis_search)
        for name in ('device','actor_device'):
            device=torch.device(getattr(self,name))
            if device.type not in ('cpu','cuda'):raise ValueError('Unsupported training device')
            if device.type=='cuda' and (not torch.cuda.is_available() or (device.index or 0)>=torch.cuda.device_count()):raise ValueError('Requested GPU unavailable')

def v3_training_identity(project,config,descriptor):
    return {'schema':V3_TRAIN_SCHEMA,'config':asdict(config),'sources':v3_source_identity(project),
            'curriculum_sha256':descriptor['sha256'],'baseline_checkpoint_sha256':config.initial_checkpoint_sha256}

def v3_read_training_checkpoint(run_dir,identity=None,selection='latest'):
    run_dir=Path(run_dir);path=run_dir/'manifest.json'
    if not path.exists():
        if list(run_dir.glob('*.pt')):raise RuntimeError('Payload exists without checkpoint manifest')
        return None,None
    manifest=json.loads(path.read_text());entries=manifest['entries']
    if not entries or len({e['file'] for e in entries})!=len(entries):raise RuntimeError('Invalid checkpoint entries')
    if manifest['latest']!=max(entries,key=lambda e:e['global_step'])['file']:raise RuntimeError('Latest checkpoint cursor mismatch')
    if manifest.get('best') not in {e['file'] for e in entries}:raise RuntimeError('Best checkpoint missing')
    loaded={}
    for entry in entries:
        state=_checkpoint_entry_state(run_dir,entry)
        if state is None or state.get('schema')!=V3_TRAIN_SCHEMA:raise RuntimeError('Missing, corrupt or wrong-schema checkpoint '+entry['file'])
        if identity is not None and state['identity']!=identity:raise RuntimeError('Source/config/data/topology identity changed; no silent restart')
        if state['optimizer_step']<0 or state['global_step']<state['optimizer_step']:raise RuntimeError('Invalid resume cursor')
        loaded[entry['file']]=(state,entry)
    if selection not in ('latest','best'):raise ValueError('Choose latest or best checkpoint')
    return loaded[manifest[selection]]

def v3_model_from_config(config,device):
    return V3AHR(base_config=config['base_config'],latent_dim=config['latent_dim'],action_dim=config['action_dim']).to(device)

def v3_new_components(project,config):
    v3_seed(config.seed)
    baseline,metadata=load_go_ahr(project,config.initial_run,device='cpu',selection=config.initial_selection)
    if metadata['checkpoint_sha256']!=config.initial_checkpoint_sha256:raise RuntimeError('Baseline initialization checkpoint changed')
    model=V3AHR(base_config=baseline.config).to(config.device)
    model.base.load_state_dict(baseline.state_dict(),strict=True)
    target=copy.deepcopy(model).eval()
    for parameter in target.parameters():parameter.requires_grad_(False)
    optimizer=torch.optim.AdamW(model.parameters(),lr=config.learning_rate,weight_decay=config.weight_decay)
    scheduler=torch.optim.lr_scheduler.ExponentialLR(optimizer,gamma=0.9999)
    scaler=torch.amp.GradScaler('cuda',enabled=False)
    return (model,target,optimizer,scheduler,scaler),metadata

def v3_pack_training(state,components):
    model,target,optimizer,scheduler,scaler=components
    state.update(model=selfplay09_cpu_tree(model.state_dict()),target_model=selfplay09_cpu_tree(target.state_dict()),
                 optimizer=selfplay09_cpu_tree(optimizer.state_dict()),scheduler=copy.deepcopy(scheduler.state_dict()),scaler=copy.deepcopy(scaler.state_dict()),rng=capture_rng())
    return state

def v3_restore_training(state,components):
    model,target,optimizer,scheduler,scaler=components
    model.load_state_dict(state['model'],strict=True);target.load_state_dict(state['target_model'],strict=True);target.eval()
    optimizer.load_state_dict(state['optimizer']);scheduler.load_state_dict(state['scheduler']);scaler.load_state_dict(state['scaler'])
    restore_rng(state['rng'])

def v3_initial_training_state(project,run_id,config,descriptor,components,metadata):
    model=components[0];weights=selfplay09_cpu_tree(model.state_dict());modelconfig={'base_config':asdict(model.config),'latent_dim':model.latent_dim,'action_dim':model.action_dim}
    return {'schema':V3_TRAIN_SCHEMA,'identity':v3_training_identity(project,config,descriptor),'config':asdict(config),'model_config':modelconfig,
            'model':weights,'target_model':selfplay09_cpu_tree(components[1].state_dict()),'optimizer':None,'scheduler':None,'scaler':None,
            'rng':capture_rng(),'global_step':0,'optimizer_step':0,'phase':'warmup' if config.warmup_updates else 'new_game',
            'warmup_remaining':config.warmup_updates,'updates_remaining':0,'games_finished':0,'current_game':None,'archive_entries':[],
            'reanalysis':[],'reanalysis_remaining':0,'reanalysis_game':None,
            'sampler':{'position':0,'epoch':0,'category_counts':{},'archive_strata':{},'horizons':{}},
            'opponents':[{'model':copy.deepcopy(weights),'digest':go_model_digest(model),'optimizer_step':0,'historical':True}],
            'initialization_digest':go_model_digest(model),'initialization':metadata,'curriculum':descriptor,
            'metrics':{'completed_games':0,'unknown_games':0,'verified_plies':0,'best_training_loss':None,'last_update':None,'loss_history':[],
                       'search_calls':0,'search_nodes':0,'deadline_exceedances':0,'reanalysis_updates':0},
            'distributed':{'world_size':1,'learner_device':config.device,'actor_device':config.actor_device,'asynchronous_actors':False},
            'run_id':run_id,'provenance':{'training_targets':'exact rules + AHR selfplay/search only','external_engine_labels':False,'purpose':config.purpose,
            'initialization_shared_between_arms':True,'unknown_prefix_policy':'transitions only; no invented outcomes','historical_replay_import':'NOT_IMPORTED: historical replay has not been reconstructed and verified as complete trajectories'}}

def v3_cached_actor(state,config,runtime,opponent_index=None):
    key='learner' if opponent_index is None else 'opponent-'+str(opponent_index)
    actors=runtime.setdefault('actors',{})
    if key not in actors:
        saved=capture_rng()
        try:
            actor=v3_model_from_config(state['model_config'],config.actor_device)
            weights=state['model'] if opponent_index is None else state['opponents'][opponent_index]['model']
            actor.load_state_dict(weights,strict=True);actor.eval();actors[key]=actor
        finally:restore_rng(saved)
    return actors[key]

def v3_behavior_move(board,actor,config):
    if config.validation_force_pass:
        policy=[0.0]*82;policy[81]=1.0
        return {'action':81,'policy_target':policy,'behavior_policy':policy,'status':'SYNTHETIC_FIXTURE','nodes':0,'elapsed_s':0.0,'model_calls':0,'fallback_pass':False}
    with torch.inference_mode():result=v3_choose_move(board,actor,time_budget_s=config.time_budget_s,search_config=V3SearchConfig(**config.search))
    policy=selfplay09_policy(board,result['policy_target']);positive=policy>0
    behavior=np.zeros_like(policy);weights=np.log(policy[positive])/config.temperature;behavior[positive]=np.exp(weights-weights.max());behavior/=behavior.sum()
    legal=list(legal_actions(board));uniform=np.zeros_like(policy);uniform[legal]=1/len(legal)
    behavior=(1-config.exploration)*behavior+config.exploration*uniform
    action=int(np.random.choice(len(policy),p=behavior))
    return {**result,'action':action,'policy_target':policy.tolist(),'behavior_policy':behavior.tolist()}

def v3_optimizer_event(project,run_id,state,components,config,curriculum,runtime):
    model,target,optimizer,scheduler,scaler=components
    examples,sequences=v3_stratified_batch(project,run_id,state,config,curriculum,runtime)
    if state['reanalysis'] and state['phase']=='learn':
        count=min(len(examples)//4,len(state['reanalysis']))
        for i in range(count):examples[i]=v3_augment_example(random.choice(state['reanalysis']),random.randrange(8))
    model.train();target.eval();optimizer.zero_grad(set_to_none=True)
    loss,metrics=v3_training_loss(model,target,examples,sequences,config.use_jepa,queen_cycles=config.queen_cycles,jepa_weight=config.jepa_weight)
    if not bool(torch.isfinite(loss)):raise RuntimeError('Nonfinite training loss')
    scaler.scale(loss).backward();scaler.unscale_(optimizer)
    norm=torch.nn.utils.clip_grad_norm_(model.parameters(),5.0,error_if_nonfinite=True)
    scaler.step(optimizer);scaler.update();scheduler.step();v3_update_ema(target,model,config.ema_decay)
    state['optimizer_step']+=1
    metrics={key:float(value.detach().cpu()) if torch.is_tensor(value) else copy.deepcopy(value) for key,value in metrics.items()}
    metrics.update(loss=float(loss.detach().cpu()),gradient_norm=float(norm.detach().cpu()),optimizer_step=state['optimizer_step'],archive_games=len(state['archive_entries']),sequence_examples=len(sequences))
    state['metrics']['last_update']=metrics;state['metrics']['loss_history']=(state['metrics']['loss_history']+[metrics])[-2000:]
    best=state['metrics']['best_training_loss'];improved=best is None or metrics['loss']<best
    if improved:state['metrics']['best_training_loss']=metrics['loss']
    return improved

def v3_training_event(project,run_id,state,components,config,curriculum,runtime):
    improved=False;model=components[0]
    if state['phase']=='warmup':
        improved=v3_optimizer_event(project,run_id,state,components,config,curriculum,runtime)
        state['warmup_remaining']-=1
        if not state['warmup_remaining']:state['phase']='new_game'
    elif state['phase']=='new_game':
        runtime.pop('actors',None)
        if state['games_finished']>=config.games:state['phase']='complete'
        else:
            state['model']=selfplay09_cpu_tree(model.state_dict())
            opponent=random.randrange(len(state['opponents'])) if random.random()<config.opponent_probability else None
            board=initial_state(config.board_size,config.komi)
            state['current_game']={'schema':V3_TRAJECTORY_SCHEMA,'split':'train','game_id':state['games_finished'],'run_id':run_id,
                'initial_state':to_dict(board),'state':to_dict(board),'records':[],'status':'IN_PROGRESS','reason':None,
                'opponent_index':opponent,'learner_color':1 if state['games_finished']%2==0 else -1,
                'learner_digest':go_model_digest(model),'purpose':config.purpose}
            state['phase']='play'
    elif state['phase']=='play':
        game=state['current_game'];board=from_dict(game['state'])
        if len(game['records'])>=config.max_plies:
            game.update(status='UNKNOWN',reason='max_plies_cutoff');state['phase']='ingest'
        else:
            opponent=game['opponent_index'] if board.to_play!=game['learner_color'] else None
            actor=v3_cached_actor(state,config,runtime,opponent);result=v3_behavior_move(board,actor,config)
            action=int(result['action']);child=play(board,action)
            if not verify_go_emission(board,action,child):raise RuntimeError('Independent selfplay emission verifier disagreed')
            game['records'].append({'state':to_dict(board),'action':action,'policy_target':result['policy_target'],'behavior_policy':result['behavior_policy'],
                'fallback_pass':bool(result.get('fallback_pass',False)),'actor_digest':go_model_digest(actor),
                'search':{'config':config.search,'nodes':int(result.get('nodes',0)),'status':result.get('status','UNKNOWN'),'time_budget_s':config.time_budget_s,'strategic_status':'UNKNOWN'}})
            game['state']=to_dict(child);state['metrics']['verified_plies']+=1;state['metrics']['search_calls']+=int(not config.validation_force_pass)
            state['metrics']['search_nodes']+=int(result.get('nodes',0));state['metrics']['deadline_exceedances']+=int(result.get('elapsed_s',0)>config.time_budget_s)
            if child.terminal:game.update(status='VERIFIED_TERMINAL',reason='two_legal_passes');state['phase']='ingest'
    elif state['phase']=='ingest':
        game=state['current_game'];entry=v3_archive_append(project,run_id,game)
        if not any(e['game_id']==entry['game_id'] for e in state['archive_entries']):state['archive_entries'].append(entry)
        state['metrics']['completed_games' if game['status']=='VERIFIED_TERMINAL' else 'unknown_games']+=1
        state['games_finished']+=1;state['updates_remaining']=config.updates_per_game
        state['reanalysis']=[];state['reanalysis_game']=entry if entry['status']=='VERIFIED_TERMINAL' else None
        state['reanalysis_remaining']=min(config.reanalysis_per_game,entry['plies']) if entry['status']=='VERIFIED_TERMINAL' and not config.validation_force_pass else 0
        state['current_game']=None;state['phase']='reanalysis' if state['reanalysis_remaining'] else 'learn'
    elif state['phase']=='reanalysis':
        game=v3_archive_read(project,run_id,state['reanalysis_game']);position=random.randrange(len(game['records']))
        example=v3_trajectory_example(game,position);board=from_dict(example['state'])
        actor=v3_cached_actor(state,config,runtime)
        with torch.inference_mode():result=v3_choose_move(board,actor,time_budget_s=config.reanalysis_time_budget_s,search_config=V3SearchConfig(**config.reanalysis_search))
        if not result.get('fallback_pass',False):
            example['policy_target']=selfplay09_policy(board,result['policy_target']).tolist();example['source']='deeper_ahr_reanalysis';example['policy_weight']=1.0
            state['reanalysis'].append(example);state['metrics']['reanalysis_updates']+=1
        state['reanalysis_remaining']-=1
        if not state['reanalysis_remaining']:state['phase']='learn'
    elif state['phase']=='learn':
        improved=v3_optimizer_event(project,run_id,state,components,config,curriculum,runtime);state['updates_remaining']-=1
        if not state['updates_remaining']:
            if state['games_finished']%config.snapshot_every_games==0:
                snapshot={'model':selfplay09_cpu_tree(model.state_dict()),'digest':go_model_digest(model),'optimizer_step':state['optimizer_step'],'historical':False}
                historical=[o for o in state['opponents'] if o['historical']];rolling=[o for o in state['opponents'] if not o['historical']]
                state['opponents']=historical+(rolling+[snapshot])[-config.opponent_snapshots:]
            state['phase']='new_game'
    elif state['phase']!='complete':raise RuntimeError('Unknown training phase '+state['phase'])
    state['global_step']+=1
    return improved

def v3_training_receipt(project,run_id,state,checkpoint_entry):
    report={'schema':V3_TRAIN_SCHEMA,'status':'COMPLETE' if state['phase']=='complete' else 'RUNNING','phase':state['phase'],'run_id':run_id,
        'global_step':state['global_step'],'optimizer_steps':state['optimizer_step'],'completed_games':state['metrics']['completed_games'],'unknown_games':state['metrics']['unknown_games'],
        'scheduled_games':state['config']['games'],'games_finished':state['games_finished'],'archive_games':len(state['archive_entries']),
        'archive_plies':sum(e['plies'] for e in state['archive_entries']),'archive_path':str(v3_archive_root(project,run_id).relative_to(Path(project))),
        'model_checkpoint':checkpoint_entry,'initialization_digest':state['initialization_digest'],'curriculum_sha256':state['curriculum']['sha256'],
        'config':state['config'],'metrics':state['metrics'],'sampler':state['sampler'],'source_identity':state['identity']['sources'],
        'updated_unix':time.time(),'checkpoint_resume':'Run same v3_run_training(project,run_id,V3TrainConfig(**config)) from original controller in tmux',
        'strength_claim':'NOT_ESTABLISHED: training loss is not promotion evidence'}
    atomic_json(v3_report_path(project,run_id+'_training.json'),report)
    return report

def v3_run_training(project,run_id,config,*,stop_after_events=None):
    project=Path(project).resolve()
    if not isinstance(config,V3TrainConfig):config=V3TrainConfig(**config)
    if stop_after_events is not None and (type(stop_after_events) is not int or stop_after_events<0):raise ValueError('Event limit must be nonnegative')
    v3_archive_root(project,run_id)
    curriculum,descriptor=v3_read_curriculum(project,config.curriculum_path)
    identity=v3_training_identity(project,config,descriptor);run_dir=project/'checkpoints'/run_id
    with checkpoint_lock(run_dir):
        state,entry=v3_read_training_checkpoint(run_dir,identity)
        components,metadata=v3_new_components(project,config)
        if state is None:
            state=v3_initial_training_state(project,run_id,config,descriptor,components,metadata)
            entry=_save_checkpoint_unlocked(run_dir,v3_pack_training(state,components),best=True,resume_cell='experiments/ahr_jepa_v3/09_trajectory_learning.ipynb: v3_run_training with unchanged original configuration')
        else:
            v3_verify_archive_references(project,run_id,state['archive_entries']);v3_restore_training(state,components)
        first=last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();runtime={}
        while state['phase']!='complete':
            if stop_after_events is not None and state['global_step']-first>=stop_after_events:break
            before_phase=state['phase'];before_epoch=state['sampler']['epoch']
            improved=v3_training_event(project,run_id,state,components,config,curriculum,runtime)
            due=(before_phase!=state['phase'] or before_epoch!=state['sampler']['epoch'] or improved or state['global_step']-last_event>=config.checkpoint_every_events or
                 state['optimizer_step']-last_update>=config.checkpoint_every_optimizer_steps or time.monotonic()-last_time>=config.checkpoint_interval_s)
            if due:
                entry=_save_checkpoint_unlocked(run_dir,v3_pack_training(state,components),best=improved,
                    resume_cell='experiments/ahr_jepa_v3/09_trajectory_learning.ipynb: v3_run_training with unchanged original configuration')
                last_event=state['global_step'];last_update=state['optimizer_step'];last_time=time.monotonic();v3_training_receipt(project,run_id,state,entry)
        if state['global_step']!=last_event:
            entry=_save_checkpoint_unlocked(run_dir,v3_pack_training(state,components),resume_cell='experiments/ahr_jepa_v3/09_trajectory_learning.ipynb: v3_run_training with unchanged original configuration')
        return v3_training_receipt(project,run_id,state,entry)

def v3_load_model(project,run_id,device='cpu',selection='latest'):
    project=Path(project);run_dir=project/'checkpoints'/run_id
    with checkpoint_lock(run_dir):
        state,entry=v3_read_training_checkpoint(run_dir,selection=selection)
        if state is None:raise FileNotFoundError('No v3 trained model '+run_id)
        if state['identity']['sources']!=v3_source_identity(project):raise RuntimeError('Model source identity changed')
        _,descriptor=v3_read_curriculum(project,state['curriculum'])
        if descriptor['sha256']!=state['identity']['curriculum_sha256']:raise RuntimeError('Training curriculum changed')
        saved=capture_rng()
        try:
            model=v3_model_from_config(state['model_config'],device);model.load_state_dict(state['model'],strict=True);model.eval()
        finally:restore_rng(saved)
    metadata={'run_id':run_id,'config':state['config'],'model_config':state['model_config'],'checkpoint_file':str((run_dir/entry['file']).relative_to(project)),
        'checkpoint_sha256':entry['sha256'],'optimizer_steps':state['optimizer_step'],'phase':state['phase'],
        'initialization_digest':state['initialization_digest'],'curriculum_sha256':state['curriculum']['sha256'],'model_sha256':go_model_digest(model),
        'source_identity':state['identity']['sources'],'provenance':state['provenance'],'games_finished':state['games_finished']}
    return model,metadata


## Reproducibility checks
The bounded recovery fixture performs real optimizer/EMA updates while using two scripted legal passes per game. Fresh processes stop mid-game and after an optimizer step, then must exactly match uninterrupted weights, optimizer/scheduler/scaler states, every RNG stream, sampler and metrics. A separate actual AHR pilot measures real search behavior. Fixed-work recovery uses ample time allowance; future moves under a tight wall-clock deadline can still depend on scheduling. That does not change exact restoration of the committed state.

In [ ]:
def v3_fresh_training_call(project,run_id,config,stop_after_events=None):
    project=Path(project);env=os.environ.copy()
    env.update(AHRV3_TRAIN_ACTION='recovery_worker',AHRV3_TRAIN_RUN=run_id,AHRV3_TRAIN_CONFIG=json.dumps(asdict(config)),
               AHRV3_TRAIN_STOP='' if stop_after_events is None else str(stop_after_events))
    out=v3_report_path(project,run_id+'-worker-'+uuid.uuid4().hex[:8]+'.ipynb');log=out.with_suffix('.log')
    command=[str(project/'.venv/bin/python'),'-m','jupyter','nbconvert','--execute','--to','notebook','--ExecutePreprocessor.timeout=-1',
             '--output',out.name,'--output-dir',str(out.parent),str(project/'experiments/ahr_jepa_v3/09_trajectory_learning.ipynb')]
    with open(log,'w') as handle:result=subprocess.run(command,cwd=project,env=env,stdout=handle,stderr=subprocess.STDOUT)
    if result.returncode:raise RuntimeError('Fresh process failed; inspect '+str(log))
    return json.loads(v3_report_path(project,run_id+'_training.json').read_text())

def v3_recovery_compare(project,left_run,right_run):
    left,_=v3_read_training_checkpoint(Path(project)/'checkpoints'/left_run)
    right,_=v3_read_training_checkpoint(Path(project)/'checkpoints'/right_run)
    keys=('model','target_model','optimizer','scheduler','scaler','rng','sampler','global_step','optimizer_step','phase','warmup_remaining',
          'updates_remaining','games_finished','current_game','opponents','metrics','initialization_digest')
    failures=[key for key in keys if not nested_equal(left[key],right[key])]
    if failures:raise AssertionError('Exact recovery mismatch: '+str(failures))
    if len(left['archive_entries'])!=len(right['archive_entries']):raise AssertionError('Archive count mismatch')
    for a,b in zip(left['archive_entries'],right['archive_entries']):
        game_a=v3_archive_read(project,left_run,a);game_b=v3_archive_read(project,right_run,b)
        game_a.pop('run_id');game_b.pop('run_id')
        if game_a!=game_b:raise AssertionError('Chronological archived game mismatch')
    return {'exact_keys':list(keys),'archived_games_equal':len(left['archive_entries']),'optimizer_steps':left['optimizer_step'],'events':left['global_step']}

def v3_validate_recovery(project,device='cpu',actor_device='cpu',*,real_search=False):
    project=Path(project);suffix=uuid.uuid4().hex[:10];prefix='ahr-v3-recovery-'+('gpu' if device.startswith('cuda') else 'cpu')+'-'+suffix
    config=V3TrainConfig(seed=270901,use_jepa=True,games=2,warmup_updates=1,updates_per_game=2,batch_size=4,device=device,actor_device=actor_device,
        max_plies=4 if real_search else 8,search={'max_depth':1,'branch_width':2,'max_nodes':4,'queen_cycles':2},time_budget_s=30.0,
        reanalysis_per_game=0,snapshot_every_games=1,purpose='synthetic_smoke',validation_force_pass=not real_search)
    runs={'full':prefix+'-full','resume':prefix+'-resume'}
    v3_fresh_training_call(project,runs['full'],config)
    v3_fresh_training_call(project,runs['resume'],config,3)
    mid,_=v3_read_training_checkpoint(project/'checkpoints'/runs['resume'])
    if mid['phase']!='play' or not mid['current_game']['records']:raise AssertionError('Did not stop mid-game')
    second_stop=6 if real_search else 3
    v3_fresh_training_call(project,runs['resume'],config,second_stop)
    after,_=v3_read_training_checkpoint(project/'checkpoints'/runs['resume'])
    if after['optimizer_step']<=mid['optimizer_step']:raise AssertionError('Did not cross optimizer event')
    v3_fresh_training_call(project,runs['resume'],config)
    comparison=v3_recovery_compare(project,runs['full'],runs['resume'])
    return {'status':'PASS','fresh_process':True,'exact_resume':True,'real_ahr_moves':real_search,'device':device,'actor_device':actor_device,
            'config':asdict(config),'runs':runs,'comparison':comparison,'source_identity':v3_source_identity(project),
            'midgame_event':mid['global_step'],'postupdate_event':after['global_step'],'scheduling_scope':'Committed state exact; real timed search may depend on subsequent scheduling'}

def v3_validate_data(project):
    project=Path(project);data,descriptor=v3_read_curriculum(project)
    if set(data['categories'])!={'capture','atari_escape','connection','ladder_bounded','life_death_bounded'}:raise AssertionError('Missing tactical categories')
    family_splits=defaultdict(set);game_splits=defaultdict(set)
    for example in data['examples']:
        family_splits[example['family_id']].add(example['split']);game_splits[example['provenance']['game_id']].add(example['split'])
        if example['family_id'] in data['excluded_old_pattern_families']:raise AssertionError('Old diagnostic pattern leaked')
        if not all(v3_tactical_success(example,a) for a in example['good_actions']):raise AssertionError('Incorrect bounded target')
        if any(k in example for k in ('outcome_target','score_target','ownership_target')):raise AssertionError('Tactical position fabricated outcome')
    if any(len(v)!=1 for v in family_splits.values()) or any(len(v)!=1 for v in game_splits.values()):raise AssertionError('Whole-family/game split leaked')
    transformed_checks=0
    for example in data['examples'][:24]:
        original=from_dict(example['state']);action=example['good_actions'][0]
        for symmetry in range(8):
            mapped=v3_action_transform(action,symmetry);transformed=transform_go(original,symmetry)
            if to_dict(play(transformed,mapped))!=to_dict(transform_go(play(original,action),symmetry)):raise AssertionError('D4 history/move transition mismatch')
            augmented=v3_augment_example(example,symmetry)
            if not v3_tactical_success(augmented,mapped):raise AssertionError('D4 tactical target mismatch')
            vector=list(range(81));ownership=v3_vector_transform(vector,symmetry)
            if sorted(ownership)!=vector:raise AssertionError('D4 ownership mapping is not permutation')
            if v3_action_transform(81,symmetry)!=81:raise AssertionError('Pass transformed incorrectly')
            transformed_checks+=1
    for sequence in data['sequences']:v3_verify_sequence(sequence)
    seqcounts=Counter((s['split'],len(s['actions'])) for s in data['sequences'])
    if any(not seqcounts[(split,h)] for split in ('train','development') for h in V3_HORIZONS):raise AssertionError('Need all prediction horizons in train/development')
    # UNKNOWN prefix is valid sequence data but cannot generate any outcome example.
    board=initial_state();child=play(board,0);policy=[0.0]*82;policy[0]=1.0
    unknown={'schema':V3_TRAJECTORY_SCHEMA,'split':'train','game_id':0,'run_id':'test','initial_state':to_dict(board),'state':to_dict(child),
             'status':'UNKNOWN','records':[{'state':to_dict(board),'action':0,'policy_target':policy,'behavior_policy':policy}]}
    v3_verify_trajectory(unknown);v3_verify_sequence(v3_trajectory_sequence(unknown,0,1))
    try:v3_trajectory_example(unknown,0)
    except ValueError:pass
    else:raise AssertionError('UNKNOWN admitted outcome supervision')
    tampered=copy.deepcopy(unknown);tampered['state']['history']=[]
    try:v3_verify_trajectory(tampered)
    except ValueError:pass
    else:raise AssertionError('Truncated superko history accepted')
    report={'status':'PASS','curriculum':descriptor,'d4_verified_transitions':transformed_checks,'whole_family_and_game_splits':True,
            'unknown_transition_only_verified':True,'history_tamper_rejected':True,'sequences':len(data['sequences']),
            'sequence_counts':{str(k):v for k,v in seqcounts.items()},'categories':data['categories'],
            'claims':'Exact bounded target checks, not Go strength or full-game solving','pattern_scope':'Five-by-five target topology; outside-board contexts are not a proof of statistical independence'}
    atomic_json(v3_report_path(project,'data_validation.json'),report)
    return report


## Run bounded checks or an explicit controller stage
Run All defaults to curriculum generation and data checks. Set `AHRV3_TRAIN_ACTION=validate` for fresh-process CPU and production-topology recovery; `recovery_worker` is internal. Actual full training is dispatched only by the frozen experiment controller. Existing original notebooks and trained checkpoints are never overwritten.

In [ ]:
ACTION=os.environ.get('AHRV3_TRAIN_ACTION','data')
if ACTION=='recovery_worker':
    cfg=V3TrainConfig(**json.loads(os.environ['AHRV3_TRAIN_CONFIG']))
    stop=os.environ.get('AHRV3_TRAIN_STOP','')
    receipt=v3_run_training(PROJECT,os.environ['AHRV3_TRAIN_RUN'],cfg,stop_after_events=int(stop) if stop else None)
    print(json.dumps({k:receipt[k] for k in ('status','run_id','optimizer_steps','completed_games','unknown_games','phase')},indent=2))
elif ACTION in ('data','validate'):
    descriptor=v3_generate_curriculum(PROJECT)
    data_receipt=v3_validate_data(PROJECT)
    atomic_json(v3_report_path(PROJECT,'curriculum_progress.json'),{'status':'COMPLETE','examples':descriptor['examples'],'categories':descriptor['categories'],
        'sequences':descriptor['sequences'],'sha256':descriptor['sha256'],'collection_bound':descriptor['generation_config']['max_games'],
        'claims':'Actual case counts are reported; bounded collection does not imply every target quota was filled'})
    receipt={'status':'PASS','data':data_receipt,'training_launched':False}
    if ACTION=='validate':
        receipt['cpu_recovery']=v3_validate_recovery(PROJECT)
        receipt['cpu_real_ahr_recovery']=v3_validate_recovery(PROJECT,real_search=True)
        if torch.cuda.is_available() and torch.cuda.device_count()>=2:
            receipt['gpu_scripted_recovery']=v3_validate_recovery(PROJECT,'cuda:0','cuda:1')
            receipt['production_recovery']=v3_validate_recovery(PROJECT,'cuda:0','cuda:1',real_search=True)
            receipt['production_recovery']['scripted_recovery']=receipt['gpu_scripted_recovery']
            coverage_run=receipt['gpu_scripted_recovery']['runs']['full']
        else:
            coverage_run=receipt['cpu_recovery']['runs']['full']
        # Exercise the actual deeper AHR teacher and a following optimizer update
        # on a disposable copy of a verified terminal fixture. Original run files
        # remain unchanged and synthetic runs cannot enter strength comparisons.
        fixture,fixture_entry=v3_read_training_checkpoint(PROJECT/'checkpoints'/coverage_run)
        original_digest=file_sha256(PROJECT/'checkpoints'/coverage_run/fixture_entry['file'])
        fixture=copy.deepcopy(fixture)
        coverage_config={**fixture['config'],'reanalysis_per_game':1,'reanalysis_time_budget_s':30.0,
            'reanalysis_search':{'max_depth':2,'max_nodes':16,'branch_width':4,'queen_cycles':2}}
        coverage_config=V3TrainConfig(**coverage_config)
        components,_=v3_new_components(PROJECT,coverage_config);v3_restore_training(fixture,components)
        fixture.update(phase='reanalysis',reanalysis_game=fixture['archive_entries'][0],reanalysis_remaining=1,reanalysis=[],updates_remaining=1)
        data,_=v3_read_curriculum(PROJECT);runtime={};updates_before=fixture['optimizer_step']
        v3_training_event(PROJECT,coverage_run,fixture,components,coverage_config,data,runtime)
        assert fixture['phase']=='learn' and fixture['reanalysis_remaining']==0 and len(fixture['reanalysis'])==1
        teacher_example=fixture['reanalysis'][0]
        assert teacher_example['source']=='deeper_ahr_reanalysis' and teacher_example['terminal_targets_verified']
        selfplay09_policy(from_dict(teacher_example['state']),teacher_example['policy_target'])
        final_board=from_dict(v3_archive_read(PROJECT,coverage_run,fixture['reanalysis_game'])['state'])
        exact_targets=terminal_go_targets(final_board,from_dict(teacher_example['state']).to_play)
        assert all(teacher_example[key]==value for key,value in exact_targets.items())
        v3_training_event(PROJECT,coverage_run,fixture,components,coverage_config,data,runtime)
        assert fixture['optimizer_step']==updates_before+1 and fixture['phase']=='new_game'
        assert file_sha256(PROJECT/'checkpoints'/coverage_run/fixture_entry['file'])==original_digest
        receipt['reanalysis_update']={'status':'PASS','actual_ahr_teacher':True,'following_optimizer_update':True,
            'verified_terminal_targets_preserved':True,'original_checkpoint_unchanged':True,'device':coverage_config.device,
            'actor_device':coverage_config.actor_device,'source_identity':v3_source_identity(PROJECT),'candidate_promotion_allowed':False}
        # Strict corruption and single-writer checks use an isolated copied run.
        import shutil
        audit_dir=PROJECT/'checkpoints'/('ahr-v3-corruption-audit-'+uuid.uuid4().hex[:10])
        shutil.copytree(PROJECT/'checkpoints'/coverage_run,audit_dir)
        bad_manifest=json.loads((audit_dir/'manifest.json').read_text())
        with open(audit_dir/bad_manifest['latest'],'ab') as handle:handle.write(b'corruption-audit')
        try:v3_read_training_checkpoint(audit_dir)
        except RuntimeError:corruption_rejected=True
        else:raise AssertionError('Corrupt latest checkpoint was accepted')
        lock_dir=PROJECT/'checkpoints'/('ahr-v3-lock-audit-'+uuid.uuid4().hex[:10])
        with checkpoint_lock(lock_dir):
            try:
                with checkpoint_lock(lock_dir):raise AssertionError('Second writer was accepted')
            except RuntimeError:second_writer_rejected=True
        receipt['negative_recovery_checks']={'corrupt_checkpoint_rejected':corruption_rejected,'second_writer_rejected':second_writer_rejected,
             'isolated_corruption_audit':str(audit_dir.relative_to(PROJECT))}
        if 'production_recovery' in receipt:
            receipt['production_recovery']['reanalysis_update']=receipt['reanalysis_update']
            receipt['production_recovery']['negative_recovery_checks']=receipt['negative_recovery_checks']
            atomic_json(v3_report_path(PROJECT,'production_recovery.json'),receipt['production_recovery'])
        atomic_json(v3_report_path(PROJECT,'trainer_validation.json'),receipt)
    print(json.dumps({'status':receipt['status'],'curriculum_examples':descriptor['examples'],'categories':descriptor['categories'],
                      'sequences':descriptor['sequences'],'checks':list(receipt),'training_launched':False},indent=2))
else:raise ValueError('Unknown AHRV3_TRAIN_ACTION '+ACTION)
